# NEMS price history

Half-hourly USEP, demand, reserve and regulation prices from `data/raw/nems/`.
Fetch or refresh the data with `uv run nems download`. Columns are referred to through the `Col` enum (`Col.USEP`, ...), never bare strings.

In [ ]:
import plotly.express as px
import polars as pl
from plotly.subplots import make_subplots

from battery_optimisation.columns import Col
from battery_optimisation.data.nems import load_nems
from battery_optimisation.data.predispatch import Feed, load_predispatch
from battery_optimisation.logs import configure_logging

configure_logging()
df = load_nems()  # one row per 30-min interval, keyed by Col.INTERVAL_START (Asia/Singapore)
df.shape

In [ ]:
df.tail()

## Daily averages

Daily means keep the plots light. RUSEP (the uncapped price) only exists from mid-2023, when the temporary price cap started.

In [ ]:
daily = df.group_by_dynamic(Col.INTERVAL_START, every="1d").agg(
    pl.col(
        Col.USEP, Col.RUSEP, Col.DEMAND_MW, Col.REGULATION, Col.PRIMARY_A, Col.CONTINGENCY_A
    ).mean()
)
px.line(
    daily, x=Col.INTERVAL_START, y=[Col.USEP, Col.RUSEP], title="Energy price, daily mean (SGD/MWh)"
)

In [ ]:
px.line(
    daily,
    x=Col.INTERVAL_START,
    y=[Col.REGULATION, Col.PRIMARY_A, Col.CONTINGENCY_A],
    title="Ancillary prices, daily mean (SGD/MWh)",
)

## Last two weeks, half-hourly

In [ ]:
recent = df.filter(
    pl.col(Col.INTERVAL_START) >= pl.col(Col.INTERVAL_START).max() - pl.duration(days=14)
)
px.line(recent, x=Col.INTERVAL_START, y=Col.USEP, title="USEP, half-hourly (SGD/MWh)")

## Pre-dispatch forecasts

Archived every 30 min by cron (`crontab/predispatch.sh`) in `data/raw/predispatch/`. Unlike the history, there is one row per **(run, interval)**: each run (`Col.PUBLISHED_AT`) re-forecasts the coming intervals. `Col.LEAD_PERIODS` >= 1 marks a forecast (1 = next half-hour); <= 0 are realised/current intervals included in the same file. Feeds: `Feed.ENERGY` (USEP, demand, ...) and `Feed.ANCILLARY` (reserve and regulation prices).

In [ ]:
fc = load_predispatch(Feed.ENERGY)
print(
    fc[Col.PUBLISHED_AT].n_unique(),
    "runs archived,",
    fc[Col.PUBLISHED_AT].min(),
    "->",
    fc[Col.PUBLISHED_AT].max(),
)
fc.filter(pl.col(Col.LEAD_PERIODS) >= 1).head()

How the USEP forecast changes from run to run (last 6 runs, forecast part only).

In [ ]:
last_runs = fc[Col.PUBLISHED_AT].unique().sort().tail(6)
recent_fc = fc.filter(
    pl.col(Col.PUBLISHED_AT).is_in(last_runs.implode()), pl.col(Col.LEAD_PERIODS) >= 1
)
px.line(
    recent_fc.with_columns(pl.col(Col.PUBLISHED_AT).dt.strftime("%d %b %H:%M").alias("run")),
    x=Col.INTERVAL_START,
    y=Col.USEP,
    color="run",  # plot-only label
    title="Pre-dispatch USEP by run (SGD/MWh)",
)

Forecast vs realised at a fixed lead. Realised USEP comes from the history (`df`), which is published about a week late, so this stays empty for the archive's first week.

In [ ]:
lead = 2  # periods ahead (2 = 1 hour)
usep_forecast = "usep_forecast"  # plot-only label
cmp = (
    fc.filter(pl.col(Col.LEAD_PERIODS) == lead)
    .select(Col.INTERVAL_START, pl.col(Col.USEP).alias(usep_forecast))
    .join(df.select(Col.INTERVAL_START, Col.USEP), on=Col.INTERVAL_START, how="inner")
)
if cmp.is_empty():
    print("No overlap with realised history yet; re-run `uv run nems download` in a week.")
else:
    mae = (cmp[usep_forecast] - cmp[Col.USEP]).abs().mean()
    print(f"{cmp.height} intervals, MAE at lead {lead}: {mae:.1f} SGD/MWh")
    display(px.line(cmp, x=Col.INTERVAL_START, y=[Col.USEP, usep_forecast]))

## Weather (as-of ECMWF IFS forecasts)

The weather the forecast model sees: for each day, the latest model run that was published before that day's issue time (`config/forecast.yaml`, section `weather`). Each step of the processing is reported below. Download runs with `uv run weather download`.

In [ ]:
from datetime import date

from battery_optimisation.config import WeatherVar, load_forecast
from battery_optimisation.data import weather

fcfg = load_forecast()
wx_start, wx_end = date(2025, 11, 1), df[Col.INTERVAL_START].max().date()
raw = weather.load_runs(fcfg.weather.model, weather.needed_runs(wx_start, wx_end, fcfg))
wx, wx_report = weather.to_features(raw, fcfg, wx_start, wx_end)
pl.DataFrame(
    [
        {
            "step": s.name,
            "rows_in": s.rows_in,
            "rows_out": s.rows_out,
            "filled": sum(s.values_filled.values()),
            "still_missing": sum(s.values_missing_after.values()),
            "detail": s.detail,
        }
        for s in wx_report.steps
    ]
)

Grid mean of each variable (for plotting only: the model gets every grid point).

In [ ]:
def grid_mean(var: WeatherVar) -> pl.Expr:
    return pl.mean_horizontal([weather.feature_name(var, p) for p in fcfg.weather.points]).alias(
        str(var)
    )


wx_mean = wx.select(
    Col.INTERVAL_START, Col.WEATHER_LEAD_H, *[grid_mean(v) for v in fcfg.weather.variables]
)
wx_daily = wx_mean.group_by_dynamic(Col.INTERVAL_START, every="1d").agg(
    pl.exclude(Col.INTERVAL_START).mean()
)
fig = make_subplots(
    rows=len(fcfg.weather.variables),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=[str(v) for v in fcfg.weather.variables],
)
for i, v in enumerate(fcfg.weather.variables, start=1):
    fig.add_scatter(
        x=wx_daily[Col.INTERVAL_START], y=wx_daily[str(v)], mode="lines", name=str(v), row=i, col=1
    )
fig.update_layout(
    height=180 * len(fcfg.weather.variables),
    showlegend=False,
    title="Daily mean of as-of forecasts",
)

### USEP against weather

Half-hourly points (a random sample, for speed), coloured by hour of day: much of the apparent weather effect is the daily cycle.

In [ ]:
joined = (
    df.select(Col.INTERVAL_START, Col.USEP)
    .join(wx_mean, on=Col.INTERVAL_START, how="inner")
    .drop_nulls()
)
sample = joined.sample(n=min(6000, joined.height), seed=0).with_columns(
    (pl.col(Col.INTERVAL_START).dt.hour() + pl.col(Col.INTERVAL_START).dt.minute() / 60).alias(
        "hour"
    )
)
variables = list(fcfg.weather.variables)
fig = make_subplots(
    rows=1, cols=len(variables), shared_yaxes=True, subplot_titles=[str(v) for v in variables]
)
for i, v in enumerate(variables, start=1):
    fig.add_scattergl(
        x=sample[str(v)],
        y=sample[Col.USEP],
        mode="markers",
        row=1,
        col=i,
        marker={
            "size": 3,
            "opacity": 0.4,
            "color": sample["hour"],
            "colorscale": "Twilight",
            "showscale": i == len(variables),
            "colorbar": {"title": "hour"},
        },
    )
fig.update_yaxes(title_text="USEP (SGD/MWh)", row=1, col=1)
fig.update_layout(height=420, showlegend=False, title="USEP vs as-of weather forecast (grid mean)")

In [ ]:
# Rank correlation of USEP with each variable (Spearman), overall
joined.select(pl.corr(Col.USEP, str(v), method="spearman").alias(str(v)) for v in variables)